# Modelos de ensamble: Bagging, Random Forest y Extra Trees

Los **modelos de ensamble** combinan varios modelos para mejorar el rendimiento. Esta familia (**averaging**) entrena estimadores de forma independiente y promedia sus predicciones, reduciendo el **error de varianza**.

- **Bagging** (Bootstrap Aggregation): entrena N modelos sobre N datasets generados por *bootstrap* (muestreo con reemplazo) y promedia (regresion) o vota (clasificacion).
- **Random Forest**: bagging de arboles de decision + en cada nodo se considera solo un subconjunto aleatorio de features, lo que **descorrelaciona** los arboles.
- **Extra Trees** (Extremely Randomized Trees): como Random Forest pero ademas elige los puntos de corte al azar; agrega mas aleatoriedad.

Dataset `Hitters`: predecimos el **log del salario** de jugadores de beisbol a partir de sus estadisticas.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## Datos

Cargamos `Hitters` y descartamos las filas con nulos (`Salary` faltante). De 322 filas quedan 263 completas.

In [ ]:
data_raw = pd.read_csv("../datasets/Hitters.csv")
data_complete = data_raw.dropna()
print(data_raw.shape)
print(data_complete.shape)

In [ ]:
data_complete.head()

Nos quedamos con las variables numericas (dejamos afuera las categoricas `League`, `Division`, `NewLeague`).

In [ ]:
data_columns=['AtBat','Hits','HmRun','Runs','RBI','Walks', 'Years', 'CAtBat', 'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks',
                'PutOuts', 'Assists', 'Errors', 'Salary']

data =data_complete.loc[:,data_columns]
print(data.shape)

data.head()

La variable objetivo es `Salary`. Aplicamos `log` para reducir la asimetria de la distribucion de salarios. Separamos en train/test.

In [ ]:
X = data.drop("Salary", axis=1)
print(X.shape)

y = np.log(data.Salary)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=127)

In [ ]:
scaler = StandardScaler()
X_train_scl = scaler.fit_transform(X_train)
X_test_scl = scaler.transform(X_test)

## Modelo base: regresion lineal

Entrenamos una regresion lineal comun como referencia. Reportamos el MSE sobre test.

In [ ]:
base_regressor = LinearRegression()

In [ ]:
fit_base = base_regressor.fit(X_train_scl, y_train)
predict_base = fit_base.predict(X_test_scl)
performance_base = mean_squared_error(y_test, predict_base)

In [ ]:
print(performance_base)

## Bagging

`BaggingRegressor` entrena 1000 regresiones lineales, cada una sobre un bootstrap del 30% de las filas (`max_samples=0.3`) y con solo 2 features (`max_features=2`). La prediccion final es el promedio. `n_jobs=-1` paraleliza.

In [ ]:
bag_linreg = BaggingRegressor(estimator=base_regressor, n_estimators=1000, max_samples=0.3, bootstrap=True,
                                max_features=2, bootstrap_features=False, n_jobs=-1, random_state=127)

In [ ]:
bag_linreg.fit(X_train_scl, y_train)

In [ ]:
prediction = bag_linreg.predict(X_test_scl)
performance = mean_squared_error(y_test, prediction)
print(performance)

## Random Forest

Bagging de arboles de decision con seleccion aleatoria de features en cada nodo. `max_depth=4` limita la profundidad de cada arbol.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
random_forest = RandomForestRegressor(n_estimators=1000, criterion='squared_error', max_depth=4,
                                        bootstrap=True, n_jobs=-1, random_state=127, max_samples=0.3)

In [ ]:
random_forest.fit(X_train_scl, y_train)

In [ ]:
prediction = random_forest.predict(X_test_scl)
performance = mean_squared_error(y_test, prediction)
performance

## Extra Trees

Variante que ademas elige los puntos de corte al azar (mas aleatoriedad, menos varianza).

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

In [ ]:
extra_tree = ExtraTreesRegressor(n_estimators=1000, criterion='squared_error', max_depth=5,
                                 bootstrap=True, n_jobs=-1, random_state=171, max_samples=0.3)

In [ ]:
extra_tree.fit(X_train_scl, y_train)

In [ ]:
prediction = extra_tree.predict(X_test_scl)
performance = mean_squared_error(y_test, prediction)
performance

## Comparacion

MSE sobre test (menor es mejor):

| Modelo | MSE |
|--------|-----|
| Regresion lineal (base) | ~0.414 |
| Bagging de regresion lineal | ~0.364 |
| Random Forest | ~0.205 |
| Extra Trees | ~0.207 |

Baggear la regresion lineal ayuda poco (el modelo base tiene sesgo, no varianza: promediar modelos parecidos no aporta mucho). Los **ensambles de arboles** (Random Forest, Extra Trees) reducen el MSE casi a la mitad: los arboles son de alta varianza y el promedio de muchos arboles descorrelacionados aprovecha justamente eso.